In [ ]:
import pandas as pd

# Load the single source of truth into memory
manifest_df = pd.read_csv("../data/processed/manifest.csv")

# Take a peek to make sure it loaded correctly
manifest_df.head()

,filepath,filename,split,label_name,extension,file_size_bytes,valid_extension,non_empty,is_readable,width,height,mode,file_hash,expected_mode,is_duplicate,is_valid,label,processed_filepath
0,..\data\raw\test\def_front\cast_def_0_1059.jpeg,cast_def_0_1059.jpeg,test,def_front,.jpeg,10701,True,True,True,300,300,RGB,4e26d6cf05102659baa357b0405d3b97,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
1,..\data\raw\test\def_front\cast_def_0_1063.jpeg,cast_def_0_1063.jpeg,test,def_front,.jpeg,9656,True,True,True,300,300,RGB,110f4382c2e4897c5ef7405e15de1946,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
2,..\data\raw\test\def_front\cast_def_0_108.jpeg,cast_def_0_108.jpeg,test,def_front,.jpeg,11100,True,True,True,300,300,RGB,d8700301b355952e83eb3620218b0652,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
3,..\data\raw\test\def_front\cast_def_0_1096.jpeg,cast_def_0_1096.jpeg,test,def_front,.jpeg,9797,True,True,True,300,300,RGB,695f2a02f07c7f6831f3818bca2e7909,False,False,True,1,..\data\processed\test\def_front\cast_def_0_10...
4,..\data\raw\test\def_front\cast_def_0_112.jpeg,cast_def_0_112.jpeg,test,def_front,.jpeg,11642,True,True,True,300,300,RGB,b22cbc17c6f36f3cb9192ea01e23c08b,False,False,True,1,..\data\processed\test\def_front\cast_def_0_11...


In [2]:
import torch
print(f"PyTorch Version: {torch.__version__}")

PyTorch Version: 2.13.0+cpu


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd

# 1. Load the Manifest (Filtering for Train and Test)
manifest_df = pd.read_csv("../data/processed/manifest.csv")
train_df = manifest_df[manifest_df['split'] == 'train']
test_df = manifest_df[manifest_df['split'] == 'test']

# 2. Define the Dataset Class (The "Data Loader" Blueprint)
class CastingDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Grab the file path and open the image
        img_path = self.dataframe.iloc[idx]['processed_filepath']
        image = Image.open(img_path).convert('RGB') 
        
        # Grab the label
        label = self.dataframe.iloc[idx]['label']

        # Convert image to mathematical tensor
        if self.transform:
            image = self.transform(image)

        # Return the image and label as PyTorch tensors
        return image, torch.tensor(label, dtype=torch.float32)

# 3. Apply standard PyTorch Transformations
# (Converts images to Tensors and normalizes pixel values for pre-trained models)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 4. Initialize the Loaders (Batches of 32)
train_dataset = CastingDataset(train_df, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = CastingDataset(test_df, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ==========================================
# 5. TRANSFER LEARNING: Load the Expert Model
# ==========================================

# Download a pre-trained ResNet18 model
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze the foundation (so it doesn't forget how to see shapes/edges)
for param in model.parameters():
    param.requires_grad = False

# Chop off the final prediction layer and replace it with a new one
# ResNet usually predicts 1000 classes; we only need 1 output (Defective or OK)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_features, 1),
    nn.Sigmoid() # Squashes the final math output into a probability between 0 and 1
)

print("Data Loaders ready and ResNet18 Model initialized successfully!")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\G689507/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:22<00:00, 2.04MB/s]

Data Loaders ready and ResNet18 Model initialized successfully!


In [4]:
import torch.optim as optim
import torch.nn as nn
import mlflow

# ==========================================
# 1. SETUP THE LEARNING RULES
# ==========================================
# Binary Cross Entropy Loss: The math used to grade binary (0 or 1) guesses
criterion = nn.BCELoss() 

# The Optimizer: The tool that updates the model based on the grade
# Notice we are ONLY letting it update the new final layer (model.fc), not the frozen core!
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# ==========================================
# 2. START THE MLFLOW EXPERIMENT & TRAINING LOOP
# ==========================================
mlflow.set_experiment("Flavor_B_Transfer_Learning")

# Start a tracking run
with mlflow.start_run(run_name="ResNet18_Base_Run"):
    
    # Log our hyperparameters so we don't forget them later
    epochs = 3  # We will just do 3 passes through the dataset to start
    mlflow.log_param("model_architecture", "ResNet18")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epochs", epochs)
    
    print(f"Starting Training for {epochs} Epochs on CPU...")

    for epoch in range(epochs):
        model.train() # Tell PyTorch the model is in learning mode
        running_loss = 0.0
        correct = 0
        total = 0

        # Loop through the data loader (fetching 32 images at a time)
        for images, labels in train_loader:
            
            # A. Clear out the math from the previous batch
            optimizer.zero_grad()

            # B. Forward Pass: The model looks at the images and makes its guesses
            outputs = model(images)
            
            # C. Calculate the Error (Loss): Compare guesses to actual labels
            loss = criterion(outputs.squeeze(), labels)

            # D. Backward Pass: The model learns and updates its weights
            loss.backward()
            optimizer.step()

            # --- Tracking Math ---
            running_loss += loss.item()
            # If the model's probability is > 50%, it guesses 1 (Defective)
            predictions = (outputs.squeeze() > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        # Calculate how well the model did this epoch
        epoch_loss = running_loss / len(train_loader)
        epoch_accuracy = correct / total

        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_accuracy:.2%}")
        
        # Log these metrics to the MLflow dashboard!
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        mlflow.log_metric("train_accuracy", epoch_accuracy, step=epoch)

    print("\nTraining Complete! You have successfully trained your first transfer learning model.")

c:\Users\G689507\ml-engineering-miniproject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/08 20:14:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/08 20:14:58 INFO mlflow.store.db.utils: Updating database tables
2026/08/08 20:15:00 INFO mlflow.tracking.fluent: Experiment with name 'Flavor_B_Transfer_Learning' does not exist. Creating a new experiment.


Starting Training for 3 Epochs on CPU...
Epoch 1/3 | Loss: 0.3538 | Accuracy: 87.84%
Epoch 2/3 | Loss: 0.1868 | Accuracy: 95.08%
Epoch 3/3 | Loss: 0.1488 | Accuracy: 96.24%

Training Complete! You have successfully trained your first transfer learning model.


In [5]:
# 1. Put the model in evaluation mode (turns off learning)
model.eval()

correct = 0
total = 0

print("Evaluating model on the unseen test dataset...")

# 2. Turn off the gradient engine (saves memory since we aren't updating weights)
with torch.no_grad():
    for images, labels in test_loader:
        
        # Make guesses on the test images
        outputs = model(images)
        predictions = (outputs.squeeze() > 0.5).float()
        
        # Tally up the correct guesses
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

# 3. Calculate and log the final score
test_accuracy = correct / total
print(f"Final Test Accuracy: {test_accuracy:.2%}")

# Log it to MLflow so Person 2 can see your final score!
with mlflow.start_run(run_name="ResNet18_Base_Run", nested=True):
    mlflow.log_metric("final_test_accuracy", test_accuracy)

Evaluating model on the unseen test dataset...
Final Test Accuracy: 98.32%


In [6]:
model.eval()

actual_defects = 0
defects_caught = 0

print("Calculating defect detection rates...")

with torch.no_grad():
    for images, labels in test_loader:
        
        outputs = model(images)
        predictions = (outputs.squeeze() > 0.5).float()
        
        # 1. Count how many images in this batch are ACTUALLY defective (label == 1)
        actual_defects += (labels == 1).sum().item()
        
        # 2. Count how many the model correctly PREDICTED as defective
        # (Where prediction is 1 AND label is 1)
        defects_caught += ((predictions == 1) & (labels == 1)).sum().item()

# Calculate the final catch rate
if actual_defects > 0:
    catch_rate = defects_caught / actual_defects
else:
    catch_rate = 0.0

print(f"Total Actual Defects in Test Set: {actual_defects}")
print(f"Defects Successfully Caught: {defects_caught}")
print(f"Defect Catch Rate (Recall): {catch_rate:.2%}")

Calculating defect detection rates...
Total Actual Defects in Test Set: 453
Defects Successfully Caught: 441
Defect Catch Rate (Recall): 97.35%


In [7]:
# 1. Put the model in evaluation mode
model.eval()

correct = 0
total = 0
actual_defects = 0
defects_caught = 0

print("Evaluating model and calculating all metrics...")

# 2. Turn off the gradient engine
with torch.no_grad():
    for images, labels in test_loader:
        
        outputs = model(images)
        predictions = (outputs.squeeze() > 0.5).float()
        
        # Accuracy math
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        # Recall math
        actual_defects += (labels == 1).sum().item()
        defects_caught += ((predictions == 1) & (labels == 1)).sum().item()

# 3. Calculate final scores
test_accuracy = correct / total
catch_rate = defects_caught / actual_defects if actual_defects > 0 else 0.0

print(f"Final Test Accuracy: {test_accuracy:.2%}")
print(f"Total Actual Defects: {actual_defects}")
print(f"Defects Successfully Caught: {defects_caught}")
print(f"Defect Catch Rate (Recall): {catch_rate:.2%}")

# 4. Log EVERYTHING to MLflow
# We use nested=True so it cleanly adds a new log entry to your experiment
with mlflow.start_run(run_name="ResNet18_Evaluation", nested=True):
    mlflow.log_metric("final_test_accuracy", test_accuracy)
    mlflow.log_metric("test_actual_defects", actual_defects)
    mlflow.log_metric("test_defects_caught", defects_caught)
    mlflow.log_metric("test_recall", catch_rate)
    
    print("\nSuccessfully logged all metrics to MLflow!")

Evaluating model and calculating all metrics...
Final Test Accuracy: 98.32%
Total Actual Defects: 453
Defects Successfully Caught: 441
Defect Catch Rate (Recall): 97.35%

Successfully logged all metrics to MLflow!
